# Transformer for Optiver — causal self-attention over the auction sequence

A third, architecturally-distinct model for the ensemble. Unlike the GRU/LSTM
(recurrent) it uses **causal self-attention**: each timestep attends to all
earlier steps at once. The causal mask (position `t` sees only `<= t`) is the
leakage guard — the attention analogue of `bidirectional=False`.
Trains multi-seed, saves val predictions for blending, and saves model artifacts.

In [1]:
import os, json, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
import features as F

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)
SEQ_LEN, VAL_FRAC = 55, 0.2

device: mps


In [2]:
# ── Data prep (same base 33 features as the GRU, for a comparable blend) ──
df = F.build_features(pd.read_csv("Data/train.csv"))
feat_cols = F.feature_columns(df)
dates  = np.sort(df["date_id"].unique())
cutoff = dates[-int(len(dates) * VAL_FRAC)]

X_all    = df[feat_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
scaler   = StandardScaler().fit(X_all[df["date_id"] < cutoff])
X_scaled = scaler.transform(X_all).astype(np.float32)

counts = df.groupby(["stock_id", "date_id"]).size()
assert (counts == SEQ_LEN).all(), "a stock-day isn't 55 steps long"
n_seq = len(counts)
X_seq    = X_scaled.reshape(n_seq, SEQ_LEN, len(feat_cols))
y_seq    = df["target"].to_numpy(np.float32).reshape(n_seq, SEQ_LEN)
mask_seq = (~np.isnan(y_seq)).astype(np.float32); y_seq = np.nan_to_num(y_seq, nan=0.0)

first_rows    = df.iloc[::SEQ_LEN]
seq_date      = first_rows["date_id"].to_numpy()
stock_ids     = np.sort(df["stock_id"].unique())
stock_to_idx  = {int(s): i for i, s in enumerate(stock_ids)}
seq_stock_idx = first_rows["stock_id"].map(stock_to_idx).to_numpy(np.int64)

tr = seq_date < cutoff
def to_t(idx):
    return (torch.tensor(X_seq[idx]), torch.tensor(seq_stock_idx[idx]),
            torch.tensor(y_seq[idx]), torch.tensor(mask_seq[idx]))
Xtr, Str, ytr, mtr = to_t(tr); Xva, Sva, yva, mva = to_t(~tr)
train_loader = DataLoader(TensorDataset(Xtr, Str, ytr, mtr), batch_size=256, shuffle=True)
val_loader   = DataLoader(TensorDataset(Xva, Sva, yva, mva), batch_size=512, shuffle=False)
n_features, n_stocks = len(feat_cols), len(stock_ids)
print(f"train {tr.sum():,} | val {(~tr).sum():,} | X {tuple(Xtr.shape)} | {n_features} feats, {n_stocks} stocks")

train 76,036 | val 19,200 | X (76036, 55, 33) | 33 feats, 200 stocks


In [3]:
# ── Model: causal Transformer encoder ────────────────────────────────────
class TimeTransformer(nn.Module):
    """Position t attends only to steps <= t (causal mask) -> no future leakage."""
    def __init__(self, n_features, n_stocks, emb_dim=24, d_model=128, nhead=4,
                 layers=3, dropout=0.2, seq_len=55):
        super().__init__()
        self.emb    = nn.Embedding(n_stocks, emb_dim)
        self.in_proj= nn.Linear(n_features + emb_dim, d_model)
        self.pos    = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)  # learned positions
        layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=d_model * 4,
                                           dropout=dropout, batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers)
        self.head    = nn.Sequential(nn.Linear(d_model, 64), nn.GELU(), nn.Linear(64, 1))

    def forward(self, x, stock):
        T = x.size(1)
        e = self.emb(stock).unsqueeze(1).expand(-1, T, -1)
        h = self.in_proj(torch.cat([x, e], dim=-1)) + self.pos[:, :T]
        causal = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()  # True = blocked
        h = self.encoder(h, mask=causal)
        return self.head(h).squeeze(-1)

In [4]:
# ── Loss + evaluation (masked MAE = competition metric) ──────────────────
def masked_mae(pred, y, mask):
    return (torch.abs(pred - y) * mask).sum() / mask.sum()

@torch.no_grad()
def eval_mae(model, loader):
    model.eval(); err, n = 0.0, 0.0
    for x, s, y, m in loader:
        x, s, y, m = x.to(device), s.to(device), y.to(device), m.to(device)
        p = model(x, s)
        err += (torch.abs(p - y) * m).sum().item(); n += m.sum().item()
    return err / n

val_zero = float((torch.abs(yva) * mva).sum() / mva.sum())
print("val zero-prediction MAE:", round(val_zero, 4))

val zero-prediction MAE: 6.0601


In [5]:
# ── Train one seed (grad clip + LR scheduler + early stop), save weights ──
os.makedirs("artifacts", exist_ok=True)

@torch.no_grad()
def predict_flat(model, batch=512):
    model.eval(); out = []
    for i in range(0, len(X_seq), batch):
        x = torch.tensor(X_seq[i:i+batch]).to(device)
        s = torch.tensor(seq_stock_idx[i:i+batch]).to(device)
        out.append(model(x, s).cpu().numpy())
    return np.concatenate(out).reshape(-1)

def train_one(seed, epochs=40):
    torch.manual_seed(seed); np.random.seed(seed)
    model = TimeTransformer(n_features, n_stocks).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=2)
    best, best_state, patience, bad = 1e9, None, 7, 0
    for ep in range(epochs):
        model.train()
        for x, s, y, m in train_loader:
            x, s, y, m = x.to(device), s.to(device), y.to(device), m.to(device)
            opt.zero_grad(); masked_mae(model(x, s), y, m).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        v = eval_mae(model, val_loader); sched.step(v)
        print(f"  seed {seed} epoch {ep:2d}  val MAE {v:.4f}  skill {(val_zero-v)/val_zero*100:.2f}%")
        if v < best - 1e-4:
            best, best_state, bad = v, {k: t.cpu().clone() for k, t in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    torch.save(best_state, f"artifacts/transformer_seed{seed}.pt")
    print(f"seed {seed}: BEST val MAE {best:.4f}")
    return predict_flat(model)

In [6]:
# ── Train multi-seed, save averaged val preds for blending ───────────────
seeds = [0, 1, 2]
tfm_multi = np.mean([train_one(s) for s in seeds], axis=0)

out = df[["row_id", "date_id", "seconds_in_bucket", "target"]].copy()
out["transformer"] = tfm_multi
out[out.date_id >= cutoff].to_parquet("preds_transformer_val.parquet", index=False)
print("saved preds_transformer_val.parquet + artifacts/transformer_seed*.pt")

  seed 0 epoch  0  val MAE 5.9731  skill 1.43%
  seed 0 epoch  1  val MAE 5.9655  skill 1.56%
  seed 0 epoch  2  val MAE 5.9576  skill 1.69%
  seed 0 epoch  3  val MAE 5.9543  skill 1.74%
  seed 0 epoch  4  val MAE 5.9485  skill 1.84%
  seed 0 epoch  5  val MAE 5.9498  skill 1.82%
  seed 0 epoch  6  val MAE 5.9385  skill 2.01%
  seed 0 epoch  7  val MAE 5.9358  skill 2.05%
  seed 0 epoch  8  val MAE 5.9420  skill 1.95%
  seed 0 epoch  9  val MAE 5.9400  skill 1.98%
  seed 0 epoch 10  val MAE 5.9428  skill 1.94%
  seed 0 epoch 11  val MAE 5.9350  skill 2.06%
  seed 0 epoch 12  val MAE 5.9382  skill 2.01%
  seed 0 epoch 13  val MAE 5.9375  skill 2.02%
  seed 0 epoch 14  val MAE 5.9379  skill 2.02%
  seed 0 epoch 15  val MAE 5.9404  skill 1.97%
  seed 0 epoch 16  val MAE 5.9430  skill 1.93%
  seed 0 epoch 17  val MAE 5.9473  skill 1.86%
  seed 0 epoch 18  val MAE 5.9470  skill 1.87%
seed 0: BEST val MAE 5.9350
  seed 1 epoch  0  val MAE 5.9786  skill 1.34%
  seed 1 epoch  1  val MAE 5.968

In [7]:
# ── Record transformer architecture in meta.json (for inference) ─────────
meta = json.load(open("artifacts/meta.json")) if os.path.exists("artifacts/meta.json") else {}
meta["transformer_arch"]  = {"emb_dim": 24, "d_model": 128, "nhead": 4,
                             "layers": 3, "dropout": 0.2, "seq_len": SEQ_LEN}
meta["transformer_seeds"] = seeds
meta.setdefault("feat_cols", feat_cols)
meta.setdefault("stock_to_idx", {int(k): int(v) for k, v in stock_to_idx.items()})
meta.setdefault("n_features", n_features); meta.setdefault("n_stocks", n_stocks)
json.dump(meta, open("artifacts/meta.json", "w"))
print("meta.json updated with transformer arch")

meta.json updated with transformer arch
